# MIMIC Gate MLP — Operational Pipeline (Ordered)


## Phase 1: Core Setup (Cells 1–5)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os

CONFIG = {
    "RUN_UI": False,
    "RUN_PIPELINE": False,
    "MODEL_BUNDLE_PATH": "/mnt/data/ed_phase2_model_thr_patched.joblib",
    "EVENT_LOG_PATH": str(Path("/mnt/data") / "event_log.jsonl"),
    "EQUIPMENT_STATUS_PATH": str(Path("/mnt/data") / "equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": str(Path("/mnt/data") / "equipment_moves.log"),
    "SOP_REGISTRY_PATH": str(Path("/mnt/data") / "sop_registry.json"),
    "QR_OUTPUT_DIR": str(Path("/mnt/data/qrcodes")),
}
Path(CONFIG["QR_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
print("CONFIG ready")


In [ ]:
class _FrozenConfig(dict):
    _ALLOW = {"MODEL_BUNDLE_PATH"}
    def __setitem__(self, k, v):
        if k in self._ALLOW: return super().__setitem__(k, v)
        raise RuntimeError(f"CONFIG frozen. Only allowed later change: {self._ALLOW}")
CONFIG = _FrozenConfig(CONFIG)
print("CONFIG frozen")


In [ ]:
import joblib, os
def phase2_bundle(path=None):
    path = path or CONFIG["MODEL_BUNDLE_PATH"]
    if not os.path.exists(path): return None
    try: return joblib.load(path)
    except Exception as e: print("phase2_bundle load error:", e); return None


In [ ]:
import sys, importlib.util, types, warnings, math, random
from collections import Counter, defaultdict
print("Core imports OK")


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Any, List, Optional
import pandas as pd
from pathlib import Path
import json

@dataclass
class WorkflowState:
    role: str = "ED"
    _store: Dict[str, Any] = field(default_factory=dict)
    def feature_dict(self) -> Dict[str, Any]:
        base = {
            "timestamp": self._store.get("timestamp", pd.Timestamp.utcnow().isoformat()),
            "waiting_room_count": self._store.get("waiting_room_count", 0),
            "triage_backlog": self._store.get("triage_backlog", 0),
            "since_vitals_min": self._store.get("since_vitals_min", 0),
            "current_census": self._store.get("current_census", 0),
            "staff_on_shift": self._store.get("staff_on_shift", 0),
            "staffed_beds": self._store.get("staffed_beds", 0),
            "occupied_beds": self._store.get("occupied_beds", 0),
        }
        base.update(self._store); return base
    def update_state_from_event(self, ev: Dict[str, Any]): self._store.update(ev or {})
    def apply_event_log(self, path: str):
        p = Path(path)
        if not p.exists(): return
        for line in p.read_text(encoding="utf-8").splitlines():
            try: self.update_state_from_event(json.loads(line))
            except Exception: pass

class TinyCritics:
    def score(self, ws: "WorkflowState", actions: List[Dict[str, Any]]):
        import numpy as np
        fd = ws.feature_dict()
        n = len(actions); base = 0.5; boost = 0.0
        if fd.get("since_vitals_min", 0) > 120: boost += 0.2
        if fd.get("triage_backlog", 0) > 10: boost += 0.1
        p = np.clip(base + boost, 0, 1)
        probs = np.clip(np.full(n, p, dtype=float), 0, 1)
        baseline = {"base": base, "boost": boost}
        uncertainty = {"method": "heuristic", "var": 0.05}
        return probs, baseline, uncertainty

state = WorkflowState(role="ED")
print("WorkflowState/TinyCritics ready")


## Phase 2: Core Services (Cells 6–10)


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("tracker_core", "/mnt/data/tracker_core.py")
tracker_core = importlib.util.module_from_spec(spec); spec.loader.exec_module(tracker_core)
from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry
print("tracker_core loaded")


In [ ]:
from datetime import datetime, timezone
def _append_event(ev: dict):
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False)+"\n")
    return ev


In [ ]:
def refresh_sop_registry(CONFIG, base_url: str = "https://example.invalid/sop"):
    sop = SOPRegistry(CONFIG["SOP_REGISTRY_PATH"])
    return sop.refresh(base_url=base_url)


In [ ]:
def troponin_rules(tn: float, delta: float):
    return {"elevated": bool(tn and tn>0.04), "delta_flag": bool(delta and delta>0.01)}


In [ ]:
from typing import Optional, Dict, Any
from datetime import datetime
from pathlib import Path
import json
def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"): return js
        except Exception: pass
    return {"units": [], "timestamp": datetime.utcnow().isoformat()}
def _compute_next_bed_eta(unit: Dict[str, Any]) -> Optional[int]:
    cap = int(unit.get("capacity",0) or 0); occ = int(unit.get("occupied",0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None


## Phase 3: Medical Calculators (Cells 11–15)


In [ ]:
def _pick(d: dict, *keys, default=None):
    for k in keys:
        if k in d: return d[k]
    return default
def _bool(x): return bool(x)
def _safe_round(x, n=2):
    try: return round(float(x), n)
    except Exception: return x


In [ ]:
def calc_qsofa(fd: dict) -> float:
    return float(fd.get("rr",0)>22) + float(fd.get("sbp",200)<100) + float(fd.get("gcs",15)<15)
def calc_mews(fd: dict) -> float:
    return float(fd.get("hr",0)>130) + float(fd.get("temp_c",0)>38.5)
def calc_heart(fd: dict) -> float:
    return float(fd.get("age",0)>=65) + float(fd.get("troponin",0)>0.04)


In [ ]:
def calc_pe_vte(fd: dict) -> float: return float(fd.get("d_dimer",0)>0.5)
def calc_gi_hepatic(fd: dict) -> float: return float(fd.get("inr",1.0)>1.5)


In [ ]:
def build_labs_norm(fd: dict) -> dict:
    out = {}
    if "creatinine" in fd:
        out["creatinine_stage"] = int(float(fd["creatinine"])>1.5)
    return out


In [ ]:
def phase2_bundle(fd: dict) -> dict:
    return {
        "qsofa": calc_qsofa(fd),
        "mews": calc_mews(fd),
        "heart": calc_heart(fd),
        "pe_vte": calc_pe_vte(fd),
        "gi_hepatic": calc_gi_hepatic(fd),
        **build_labs_norm(fd),
    }


## Phase 4: ML Integration (Cells 16–18)


In [ ]:
import numpy as np
def predict_one(features: dict, bundle=None, threshold: float = 0.5):
    bundle = bundle or phase2_bundle
    if callable(bundle) and bundle is phase2_bundle:
        score = float(min(1.0, max(0.0, (features.get("qsofa",0)+features.get("mews",0))/6.0)))
        return {"score": score, "label": int(score>=threshold), "threshold": threshold}
    try:
        B = phase2_bundle(features); X = np.array([list(B.values())], dtype=float)
        proba = float(bundle.predict_proba(X)[0,1])
        return {"score": proba, "label": int(proba>=threshold), "threshold": threshold}
    except Exception as e:
        return {"score": 0.0, "label": 0, "threshold": threshold, "error": str(e)}


In [ ]:
def score_patient(ws: WorkflowState, extra: dict = None):
    fd = ws.feature_dict(); fd.update(extra or {})
    b = phase2_bundle(fd)
    return {"bundle": b, "pred": predict_one({**fd, **b})}


In [ ]:
def compute_phase2_scores(rows: list) -> list:
    out = []
    for r in rows:
        b = phase2_bundle(r)
        out.append({"pred": predict_one({**r, **b}), "bundle": b})
    return out


## Phase 5: UI Components (Cells 19–22)


In [ ]:
def ui_log(msg):
    if CONFIG["RUN_UI"]: print(msg)


In [ ]:
import importlib.util
spec_ui = importlib.util.spec_from_file_location("tracker_ui", "/mnt/data/tracker_ui.py")
tracker_ui = importlib.util.module_from_spec(spec_ui); spec_ui.loader.exec_module(tracker_ui)
def run_ui():
    if not CONFIG["RUN_UI"]:
        print("RUN_UI=False — UI not launched"); return
    t = TrackerService.from_config(CONFIG)
    s = WorkflowState(role="ED")
    tracker_ui.run_ui(CONFIG, t, s)


In [ ]:
def icu_panel_load(path="/mnt/data/icu_status.json"):
    try:
        return json.loads(Path(path).read_text())
    except Exception:
        return {"units": [], "timestamp": datetime.utcnow().isoformat()}


In [ ]:
def run_calc_ui(ws: WorkflowState):
    fd = ws.feature_dict()
    b = phase2_bundle(fd)
    print("Phase-2 snapshot:", b)


## Phase 6: Extensions (Cells 23–25)


In [ ]:
def make_synth(n=5):
    import random
    rows = []
    for _ in range(n):
        rows.append({"rr": random.randint(10, 35), "sbp": random.randint(80, 160), "gcs": 15})
    return rows


In [ ]:
def emit_events(ws: WorkflowState, n=3):
    for i in range(n):
        _append_event({"kind":"tick","i":i})
    print(f"Emitted {n} events")


In [ ]:
def sanity_pack():
    import numpy as np
    from pathlib import Path
    s=WorkflowState(role="nurse"); getattr(s,"touch_now",lambda *_:None)(__import__("pandas").Timestamp.utcnow())
    tc=TinyCritics()
    p,b,u=tc.score(s,[{"id":"reassess_vitals","label":"Reassess vitals"},{"id":"order_ecg","label":"Order ECG"}])
    assert len(p)==2 and (0<=p).all() and (p<=1).all()
    from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry
    t=TrackerService.from_config(CONFIG)
    _=t.equipment_status(); t.log_move("pump-001","A1","B2"); assert Path(CONFIG["EQUIPMENT_MOVES_LOG_PATH"]).exists()
    q=QRService(CONFIG["QR_OUTPUT_DIR"]).make("poctest"); assert isinstance(q,str) and len(q)>0
    sop=SOPRegistry(CONFIG["SOP_REGISTRY_PATH"]).read(); assert sop is not None
    print("SMOKE_OK"); return True
print("Sanity cell ready")


## Add-on: Operational Feature Builder (ICU capacity–aware)


In [ ]:
from typing import Dict, Any, Optional, Sequence
import numpy as np, datetime as _dt
def _safe_div(a: float, b: float, default: float = 0.0):
    try: b = float(b); return float(a)/b if b not in (0.0,0,None) else default
    except Exception: return default
def _slope(values: Sequence[float]) -> float:
    if not values: return 0.0
    x = np.arange(len(values), dtype=float); y = np.array([float(v) for v in values], dtype=float)
    if len(y) < 2 or np.allclose(y, y[0]): return 0.0
    xm, ym = x.mean(), y.mean(); den = np.sum((x-xm)**2)
    return float(np.sum((x-xm)*(y-ym))/den) if den else 0.0
def _z_from_baseline(value, baseline):
    try:
        if value is None or baseline is None: return 0.0
        med = float(baseline.get("median",0.0)); mad = float(baseline.get("mad",0.0)); iqr = float(baseline.get("iqr",0.0))
        scale = mad if mad and mad>0 else (iqr/1.349 if iqr and iqr>0 else 0.0)
        return float((float(value)-med)/scale) if scale else 0.0
    except Exception: return 0.0
def build_operational_features(ws: WorkflowState, *, baselines=None, history=None, context=None, now=None) -> Dict[str, Any]:
    baselines = baselines or {}; history = history or {}; context = context or {}
    fd = dict(ws.feature_dict())
    ts = now or fd.get("timestamp") or _dt.datetime.utcnow().isoformat()
    if isinstance(ts, str):
        try: ts = _dt.datetime.fromisoformat(ts)
        except Exception: ts = _dt.datetime.utcnow()
    hour = int(ts.hour); dow = int(ts.weekday()); is_night = 1 if hour<7 or hour>=23 else 0
    census = fd.get("current_census") or fd.get("census") or fd.get("ed_census")
    waiting = fd.get("waiting_room_count") or 0; backlog = fd.get("triage_backlog") or 0
    los_p50 = fd.get("los_p50_min") or 0; los_p90 = fd.get("los_p90_min") or 0
    staff_on_shift = fd.get("staff_on_shift") or 0; staffed_beds = fd.get("staffed_beds") or 0; occupied_beds = fd.get("occupied_beds") or 0
    icu_capacity_total = fd.get("icu_capacity_total"); icu_occupied_beds = fd.get("icu_occupied_beds"); icu_free_beds = fd.get("icu_free_beds")
    if icu_free_beds is None and icu_capacity_total is not None and icu_occupied_beds is not None:
        try: icu_free_beds = max(0, int(icu_capacity_total) - int(icu_occupied_beds))
        except Exception: icu_free_beds = 0
    if icu_capacity_total is None and icu_occupied_beds is not None and icu_free_beds is not None:
        try: icu_capacity_total = int(icu_occupied_beds) + int(icu_free_beds)
        except Exception: icu_capacity_total = None
    icu_free_beds = int(icu_free_beds or 0); icu_occupied_beds = int(icu_occupied_beds or 0); icu_capacity_total = int(icu_capacity_total or 0)
    icu_occupancy = _safe_div(icu_occupied_beds, max(1, icu_capacity_total)); icu_free_ratio = _safe_div(icu_free_beds, max(1, icu_capacity_total))
    census_map = baselines.get("census_by_hour_dow") or {}; hist_avg = census_map.get((hour,dow)) or census_map.get(f"{hour}-{dow}") or None
    admit_rate_slope_6h = float(_slope(history.get("admit_rate_series") or []))
    icu_availability_trend = float(_slope(history.get("icu_free_beds_series") or history.get("icu_free_ratio_series") or []))
    ems_series = history.get("ems_runs_2h_series") or []; d_ems_runs_2h = float(ems_series[-1]-ems_series[-2]) if len(ems_series)>=2 else 0.0
    los_series = history.get("los_p50_series") or []; d_los_p50 = float(los_series[-1]-los_series[-2]) if len(los_series)>=2 else 0.0
    z_census_tod = float(_safe_div(census, hist_avg, 1.0)) if census is not None and hist_avg else 1.0
    queue_backlog_ratio = float(_safe_div(backlog, max(1, waiting))); staff_ratio = float(_safe_div(staff_on_shift, max(1, census or 1)))
    bed_occupancy = float(_safe_div(occupied_beds, max(1, staffed_beds or 1)))
    flu_index = float((context or {}).get("flu_index", 0.0)); is_flu_season = int((context or {}).get("is_flu_season", 0))
    specialist_coverage = float((context or {}).get("specialist_coverage", 1.0))
    imaging_tat_p50_z = _z_from_baseline(fd.get("imaging_tat_p50_min"), (baselines or {}).get("tat_img_baseline"))
    lab_tat_p50_z = _z_from_baseline(fd.get("lab_tat_p50_min"), (baselines or {}).get("tat_lab_baseline"))
    arrival_to_first_assess_p50_z = _z_from_baseline(fd.get("arrival_to_first_assess_p50_min"), (baselines or {}).get("arrival_first_assess_baseline"))
    alert_response_p50_z = _z_from_baseline(fd.get("alert_response_p50_min"), (baselines or {}).get("alert_response_baseline"))
    pressure_index = float(z_census_tod * (1.0 - staff_ratio) * bed_occupancy)
    admit_x_icu = float(max(0.0, admit_rate_slope_6h) * (1.0 / max(1.0, float(icu_free_beds))))
    eps = 1e-3; admit_x_icu_capacity = float(max(0.0, admit_rate_slope_6h) * (1.0 / max(eps, float(icu_free_ratio))))
    night_x_specialists = float(is_night * max(0.0, 1.0 - specialist_coverage)); flu_x_occupancy = float(flu_index * bed_occupancy)
    return {"hour": hour, "dow": dow, "is_night": is_night, "z_census_tod": z_census_tod,
            "waiting_room_count": int(waiting or 0), "triage_backlog": int(backlog or 0),
            "queue_backlog_ratio": queue_backlog_ratio, "los_p50_min": float(los_p50 or 0.0),
            "los_p90_min": float(los_p90 or 0.0), "d_los_p50_min": float(d_los_p50),
            "staff_ratio": staff_ratio, "bed_occupancy": bed_occupancy,
            "icu_capacity_total": int(icu_capacity_total), "icu_occupied_beds": int(icu_occupied_beds),
            "icu_free_beds": int(icu_free_beds), "icu_occupancy": float(icu_occupancy), "icu_free_ratio": float(icu_free_ratio),
            "ems_runs_2h": int(fd.get("ems_runs_2h") or 0), "d_ems_runs_2h": float(d_ems_runs_2h),
            "admit_rate_slope_6h": float(admit_rate_slope_6h), "icu_availability_trend": float(icu_availability_trend),
            "specialist_coverage": float(specialist_coverage),
            "missing_specialist_coverage": int(1 if "specialist_coverage" not in (context or {}) else 0),
            "flu_index": float(flu_index), "is_flu_season": int(is_flu_season),
            "imaging_tat_p50_z": float(imaging_tat_p50_z), "lab_tat_p50_z": float(lab_tat_p50_z),
            "arrival_to_first_assess_p50_z": float(arrival_to_first_assess_p50_z), "alert_response_p50_z": float(alert_response_p50_z),
            "pressure_index": float(pressure_index), "admit_x_icu": float(admit_x_icu),
            "admit_x_icu_capacity": float(admit_x_icu_capacity), "night_x_specialists": float(night_x_specialists),
            "flu_x_occupancy": float(flu_x_occupancy)}
def extended_feature_dict(ws: WorkflowState, *, baselines=None, history=None, context=None, now=None):
    base = dict(ws.feature_dict()); base.update(build_operational_features(ws, baselines=baselines, history=history, context=context, now=now)); return base
print("Operational builder ready.")


## Add-on: EDOperationalClassifier & ICU Bridge Integration


In [ ]:
# Import classifier from /mnt/data without executing sklearn-dependent code at import time beyond module import
import importlib.util, os, types
spec = importlib.util.spec_from_file_location("EDOperationalClassifier", "/mnt/data/EDOperationalClassifier.py")
EDOC = importlib.util.module_from_spec(spec); spec.loader.exec_module(EDOC)
from EDOperationalClassifier import EDOperationalClassifier
print("EDOperationalClassifier available")


In [ ]:
# ICU bridge import (optional)
import importlib.util, os, types
BRIDGE_PATH = "/kaggle/input/icu-bridge/operational_icu_bridge.py"
bridge = None
if os.path.exists(BRIDGE_PATH):
    spec = importlib.util.spec_from_file_location("operational_icu_bridge", BRIDGE_PATH)
    bridge = importlib.util.module_from_spec(spec); spec.loader.exec_module(bridge)
    print("ICU bridge loaded")
else:
    print("ICU bridge not found; proceeding without ICU overlay.")
# Builder handle from this notebook
builder_module = types.SimpleNamespace(extended_feature_dict=extended_feature_dict, build_operational_features=build_operational_features)
# Minimal baselines/history/context
census_map = {f"{h}-{d}": 40.0 for h in range(24) for d in range(7)}
baselines = {"census_by_hour_dow": census_map}
history  = {"admit_rate_series": [0.2, 0.25, 0.3, 0.35]}
context  = {"flu_index": 0.6, "is_flu_season": 1, "specialist_coverage": 0.7}
ws_instance = state  # live instance
X = bridge.extended_feature_dict_with_icu(ws_instance, builder_module, baselines=baselines, history=history, context=context) if bridge else extended_feature_dict(ws_instance, baselines=baselines, history=history, context=context)
print("Features built; keys:", len(X))
